# ToneFlow — full 30-stem MDB-stem-synth bench

Confirms the **93.4 % RPA** CREPE-tiny number we measured locally on the
5-stem prefix. Free-tier Colab CPU is enough; expected runtime ≈ 3.5 h
(≈ 30 min NSDF baseline + ≈ 3 h CREPE-tiny).

**Setup before running:**
1. Pack the repo locally (no node_modules / datasets / secrets):
   ```bash
   tar --exclude='./node_modules' --exclude='./.next' --exclude='./out' \
       --exclude='./bench/datasets' --exclude='./public/ort' --exclude='./.git' \
       --exclude='./.env*' --exclude='**/.DS_Store' --exclude='./.claude' \
       -czf toneflow-src.tar.gz .
   ```
   Result: ~1.9 MB.
2. Upload `toneflow-src.tar.gz` to your Google Drive at
   `MyDrive/toneflow-src.tar.gz` (drag-drop in the Drive web UI is fine).
3. **Runtime → Change runtime type → CPU.** GPU/TPU give no speedup —
   `onnxruntime-node` runs CREPE in WASM regardless and burns your
   free-tier compute units faster.
4. Runtime → Run all. Don't close the tab.

Output: two log files (`bench-mdb-nsdf.log`, `bench-mdb-crepe.log`) saved
back to Drive in the final cell.

## 1. Mount Drive + extract source

Replaces the usual `git clone` — the repo is private, so we ship the
source via Drive instead.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
TARBALL = '/content/drive/MyDrive/toneflow-src.tar.gz'
assert os.path.exists(TARBALL), (
  f'Upload toneflow-src.tar.gz to {TARBALL} first (see the cell above for '
  'the tar command).'
)

!mkdir -p /content/ToneFlow
!tar xzf '{TARBALL}' -C /content/ToneFlow
!ls /content/ToneFlow/package.json /content/ToneFlow/public/models/crepe-tiny.onnx

## 2. Install Node 20

Colab ships with Node 18; the project needs 20+ for Next 16.

In [ ]:
!curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash - >/dev/null
!sudo apt-get install -y nodejs >/dev/null
!node --version && npm --version

## 3. Install npm deps

Uses the locked tree from `package-lock.json` so versions match local.

In [ ]:
%cd /content/ToneFlow
!npm ci --no-audit --no-fund 2>&1 | tail -10

## 4. Download MDB-stem-synth (~3 GB)

Source: [Zenodo record 1481172](https://zenodo.org/records/1481172).
If it stalls partway, retry — Zenodo throttles unauthenticated traffic.
Should land 230 audio stems with matching annotations.

In [ ]:
!mkdir -p bench/datasets
!wget -q --show-progress -O /tmp/MDB-stem-synth.tar.gz \
  https://zenodo.org/records/1481172/files/MDB-stem-synth.tar.gz
!tar xzf /tmp/MDB-stem-synth.tar.gz -C bench/datasets/
!rm /tmp/MDB-stem-synth.tar.gz
!ls bench/datasets/MDB-stem-synth/audio_stems | wc -l

## 5. NSDF baseline (≈ 30 min)

`--rms-gate 0.005` matches the live adaptive-floor gate that fires in
quiet rooms. Without it, the bench reports the legacy `0.018` path
(~55 % RPA) instead of the lifted ~70 %.

In [ ]:
!npm run bench:mdb -- ./bench/datasets/MDB-stem-synth --rms-gate 0.005 \
  2>&1 | tee /content/bench-mdb-nsdf.log

## 6. CREPE-tiny bench (≈ 3 h)

Uses `public/models/crepe-tiny.onnx` (1.9 MB, shipped in the tarball).
Same resample → normalize → infer → local-weighted-mean pipeline as the
browser path.

In [ ]:
!npm run bench:mdb -- ./bench/datasets/MDB-stem-synth --rms-gate 0.005 --crepe \
  2>&1 | tee /content/bench-mdb-crepe.log

## 7. Inspect the headline numbers + save logs to Drive

Colab wipes `/content/` when the runtime disconnects, so the logs go
back to Drive.

In [ ]:
!echo '=== NSDF (baseline) ===' && tail -25 /content/bench-mdb-nsdf.log
!echo && echo '=== CREPE-tiny ===' && tail -25 /content/bench-mdb-crepe.log

!mkdir -p /content/drive/MyDrive/toneflow-bench
!cp /content/bench-mdb-*.log /content/drive/MyDrive/toneflow-bench/
!ls -la /content/drive/MyDrive/toneflow-bench/